# NBA Feature Store – Phase 1

This notebook implements a production-style data pipeline that ingests
NBA player game statistics from the NBA Stats API and stores them in a
partitioned BigQuery feature store.

The pipeline includes:

• Multi-endpoint API ingestion (traditional, advanced, usage, tracking)  
• Retry-protected API calls with exponential backoff  
• Adaptive rate limiting and session management  
• Partition-safe BigQuery ingestion  
• Schema-locked feature store design  
• Data validation and corruption protection  
• Automated health monitoring and integrity audits  

---

## Feature Store Table

pr_see_daily_player_game_log

---

## Technology Stack

Python  
nba_api  
Google BigQuery  
Pandas  

---

## Architecture Highlights

• Idempotent ingestion  
• Batch backfill engine  
• Data validation layer  
• Production-style monitoring  

---

This notebook represents **Phase 1** of a larger sports analytics system.

Phase 1 builds the **data infrastructure layer (feature store)**.

Phase 2 will introduce:

• Feature modeling  
• Player projection systems  
• Analytical modeling pipelines

# System Architecture

The pipeline follows a modern data engineering pattern for building
a reliable analytics feature store.

The ingestion process pulls NBA game data from multiple endpoints,
merges the player statistics into a unified dataset, validates the
data, and loads it into a partitioned BigQuery feature store.

## Pipeline Flow

```
        NBA Stats API
              │
              ▼
   Endpoint Pull Layer
 (traditional / advanced /
    usage / tracking)
              │
              ▼
   Merge + Feature Engineering
              │
              ▼
        Validation Layer
              │
              ▼
      BigQuery Feature Store
   (pr_see_daily_player_game_log)
              │
              ▼
   Monitoring & Integrity Audits
```

## Architecture Guarantees

• Reliable ingestion through retry + exponential backoff  
• Rate-limit protection against NBA Stats API restrictions  
• Schema-locked warehouse design preventing column drift  
• Partition-safe BigQuery writes to control query cost  
• Validation safeguards preventing corrupted data ingestion  
• Operational monitoring for pipeline health visibility

# How to Run This Notebook

This notebook is designed to run sequentially from top to bottom.
It builds and maintains the NBA player game feature store in BigQuery.

------------------------------------------------------------

STEP 1 — Authenticate Google Cloud

Run the authentication cell to connect to the configured
Google Cloud project.

This enables access to the BigQuery feature store.

------------------------------------------------------------

STEP 2 — Install Required Packages

Required Python libraries:

nba_api  
google-cloud-bigquery  
pandas-gbq  

These are installed in the setup cell at the start of the notebook.

------------------------------------------------------------

STEP 3 — Configure Runtime Settings

Runtime behavior is controlled in the **CONFIGURATION — PHASE 1** section.

Two execution modes are available.

Automatic Daily Mode

AUTO_YESTERDAY_MODE = True

The pipeline will automatically ingest the previous NBA game date
based on Eastern Time.

This is intended for daily production runs.

------------------------------------------------------------

Manual Backfill Mode

AUTO_YESTERDAY_MODE = False

START_DATE = "MM/DD/YYYY"  
END_DATE   = "MM/DD/YYYY"

This mode is used for:

• historical backfills  
• rebuilding missing partitions  
• debugging ingestion issues  

Maximum allowed range:

7 days per run

This guardrail prevents accidental large backfills.

------------------------------------------------------------

STEP 4 — Execute Notebook

Run the notebook cells sequentially.

The ingestion engine will:

• pull NBA box score data from the API  
• merge multi-endpoint statistics  
• enrich player data with roster metadata  
• perform validation checks  
• write partitioned records to BigQuery  
• verify successful ingestion

------------------------------------------------------------

Operational Monitoring

The notebook includes monitoring tools for feature store health:

• Data Health Audit — detects missing game dates or duplicate rows  
• Game Integrity Audit — verifies structural correctness of games  
• Feature Store Health Summary — ingestion freshness metrics  
• Feature Store Command Center — operational pipeline dashboard

These checks help ensure the data pipeline remains reliable and
the feature store remains consistent.

#Start of Phase 1 - Loading Game/ Player Data + Logging in BigQuery
    Currently runs and logs 10/21/2025 to 03/03/2026
    to daily_player_game_log

In [1]:
!pip install nba_api google-cloud-bigquery pandas-gbq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 6.9 MB/s eta 0:00:00


In [2]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
BQ_PROJECT_ID = "nba-structural-edge-engine"

client = bigquery.Client(project=BQ_PROJECT_ID)

print("Connected to BigQuery project:", BQ_PROJECT_ID)

Connected to BigQuery project: nba-structural-edge-engine


In [3]:
import pandas as pd
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

from nba_api.stats.endpoints import (
    scoreboardv2,
    boxscoretraditionalv3,
    boxscoreadvancedv3,
    boxscoreusagev3,
    boxscorefourfactorsv3,
    boxscoreplayertrackv3,
    boxscoresummaryv3,
    commonteamroster
)

In [4]:
# ============================================================
# CONFIGURATION — PHASE 1
# ============================================================
# PURPOSE
# ------------------------------------------------------------
# This section defines runtime configuration for the ingestion
# pipeline. It supports both:
#
# 1) AUTO_YESTERDAY_MODE
#    Automatically runs ingestion for the previous NBA
#    calendar day using Eastern Time.
#
# 2) MANUAL DATE RANGE
#    Allows explicit date ranges for historical backfills or
#    missed-day recovery.
#
# IMPORTANT DESIGN NOTES
# ------------------------------------------------------------
# • System ingestion timestamps remain UTC
# • GAME_DATE remains the NBA game calendar date
# • INGESTED_AT_UTC remains UTC system timestamp
# • No schema or table structure changes occur here
# • Only START_DATE / END_DATE assignment changes
#
# NOTE ON TIMEZONE LOGIC
# ------------------------------------------------------------
# NBA game schedules are defined relative to US/Eastern time.
# Therefore AUTO_YESTERDAY_MODE determines "yesterday" using
# Eastern time to avoid edge cases where UTC rolls into the
# next day while it is still the prior calendar day locally.
#
# This ensures:
# • Running the pipeline anytime during a given day locally
#   will always ingest the correct previous NBA game date
# • No dependency on when UTC midnight occurs
#
# This preserves:
#   - partition safety
#   - idempotent ingestion
#   - validation logic
#   - retry logic
#   - batch guardrails
#
# ============================================================


# ------------------------------------------------------------
# MODE CONTROL
# ------------------------------------------------------------
# AUTO_YESTERDAY_MODE = True
#     Pipeline automatically runs for yesterday (Eastern)
#
# AUTO_YESTERDAY_MODE = False
#     Pipeline uses manually supplied START_DATE / END_DATE
#
AUTO_YESTERDAY_MODE = True


# ------------------------------------------------------------
# BIGQUERY DESTINATION
# ------------------------------------------------------------
BQ_PROJECT_ID = "nba-structural-edge-engine"
DATASET_ID = "PR_SEE_NBA_ANALYTICS"
TABLE_NAME = "pr_see_daily_player_game_log"


# ------------------------------------------------------------
# MANUAL DATE RANGE
# ------------------------------------------------------------
# ONLY USED IF AUTO_YESTERDAY_MODE = False
#
# DO NOT EXCEED 7 DAYS PER RUN
# (Guardrail enforced elsewhere in notebook)
#
# EXAMPLE:
#
# START_DATE = "01/01/2026"
# END_DATE   = "01/07/2026"
#
START_DATE = "03/01/2026"
END_DATE   = "03/01/2026"


# ------------------------------------------------------------
# EXCLUDED DATES
# ------------------------------------------------------------
# Dates intentionally skipped during ingestion.
#
# Currently used for:
# NBA All-Star break where no standard games occur.
#
EXCLUDE_DATES = [
    "02/13/2026",
    "02/14/2026",
    "02/15/2026",
    "02/16/2026",
    "02/17/2026",
    "02/18/2026",
]


# ------------------------------------------------------------
# DERIVED TABLE IDENTIFIER
# ------------------------------------------------------------
TABLE_ID = f"{BQ_PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"


# ============================================================
# AUTO DATE CONTROLLER
# ============================================================
# When AUTO_YESTERDAY_MODE is enabled, this block overrides
# manual START_DATE and END_DATE values.
#
# Yesterday is determined relative to US/Eastern time to
# align with NBA game scheduling.
#
# Example:
#
# If Eastern time is:
#     03/03/2026 23:00
#
# Pipeline will ingest:
#     03/02/2026
#
# This prevents UTC rollover issues where UTC may already be
# the next calendar day while NBA games still belong to the
# previous schedule date.
#
# All ingestion timestamps remain stored in UTC.
#
# ============================================================

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

if AUTO_YESTERDAY_MODE:

    eastern_now = datetime.now(ZoneInfo("America/New_York"))

    yesterday_eastern = eastern_now - timedelta(days=1)

    auto_date = yesterday_eastern.strftime("%m/%d/%Y")

    START_DATE = auto_date
    END_DATE = auto_date

    print("--------------------------------------------------")
    print("[INFO] AUTO_YESTERDAY_MODE ENABLED")
    print(f"[INFO] Current Eastern Time: {eastern_now}")
    print(f"[INFO] Pipeline will run for NBA date: {START_DATE}")
    print("[INFO] Manual START_DATE / END_DATE ignored")
    print("--------------------------------------------------")

else:

    print("--------------------------------------------------")
    print("[INFO] MANUAL DATE MODE ENABLED")
    print(f"[INFO] START_DATE: {START_DATE}")
    print(f"[INFO] END_DATE:   {END_DATE}")
    print("--------------------------------------------------")

--------------------------------------------------
[INFO] AUTO_YESTERDAY_MODE ENABLED
[INFO] Current Eastern Time: 2026-03-04 14:14:01.721540-05:00
[INFO] Pipeline will run for NBA date: 03/03/2026
[INFO] Manual START_DATE / END_DATE ignored
--------------------------------------------------


In [5]:
MAX_DAYS_PER_RUN = 7  # Production safety guardrail

In [6]:
# ============================================================
# PHASE 1 SCHEMA LOCK — VERSION 1.0
# NBA FEATURE STORE
# ============================================================

"""
PHASE 1 ARCHITECTURE OVERVIEW
============================================================

This notebook implements the Phase 1 version of the NBA
Feature Store ingestion pipeline.

The schema and ingestion logic are now considered LOCKED
for Version 1.0. Any structural schema changes require
a version bump.

------------------------------------------------------------
TABLE STRUCTURE
------------------------------------------------------------

Table:
    pr_see_daily_player_game_log

Partitioning:
    GAME_DATE (DATE)

Clustering:
    PLAYER_ID
    TEAM_ID

Partition filtering is REQUIRED:
    require_partition_filter = TRUE

This prevents accidental full-table scans and protects
against runaway BigQuery costs.

------------------------------------------------------------
COLUMN GROUPS
------------------------------------------------------------

IDENTITY
    PLAYER_ID
    PLAYER_NAME
    TEAM_ID
    TEAM_TRICODE
    GAME_ID
    ROW_KEY

GAME CONTEXT
    GAME_DATE
    GAME_STATUS
    GAME_TIME_UTC
    ATTENDANCE
    ARENA_NAME
    ARENA_CITY
    ARENA_STATE
    ARENA_COUNTRY

ROSTER METADATA
    POSITION
    HEIGHT
    WEIGHT
    EXP

MINUTES ARCHITECTURE
    minutes
    minutes_SECONDS
    DNP_FLAG
    GAMES_PLAYED_FLAG

PLAYER TRADITIONAL STATS
    points, assists, rebounds, etc.

PLAYER ADVANCED STATS
    *_ADV columns

PLAYER USAGE STATS
    *_USAGE columns

PLAYER TRACKING STATS
    *_TRACK columns

TEAM CONTEXT FEATURES
    *_TEAM columns

SYSTEM METADATA
    INGESTED_AT_UTC
    LOAD_BATCH_ID

------------------------------------------------------------
CANONICAL MINUTES DESIGN
------------------------------------------------------------

Raw minutes are stored as:

    minutes (MM:SS)

Normalized numeric value:

    minutes_SECONDS

Derived flags:

    DNP_FLAG
    GAMES_PLAYED_FLAG

This prevents ambiguity when modeling player participation.

------------------------------------------------------------
INGESTION GUARANTEES
------------------------------------------------------------

The ingestion engine enforces:

• Partition-safe deletes
• Idempotent daily rebuilds
• Retry-protected API calls
• Validation before BigQuery load
• Atomic day-level ingestion
• Batch backfill protection
• Rate limiting and cooldowns

------------------------------------------------------------
VALIDATION SAFEGUARDS
------------------------------------------------------------

Before ingestion:

• Duplicate ROW_KEY detection
• Duplicate column detection
• Null PLAYER_ID rejection
• Negative minutes rejection
• Empty dataframe rejection

Post-ingestion:

• Partition row count verification

------------------------------------------------------------
PIPELINE HEALTH MONITORING
------------------------------------------------------------

Operational health checks include:

DATA HEALTH AUDIT
    - Missing regular season dates
    - Duplicate row keys

GAME INTEGRITY AUDIT
    - Missing teams
    - Suspicious player counts
    - Corrupted rows

FEATURE STORE COMMAND CENTER
    - Ingestion freshness
    - Total rows
    - Total games
    - Unique players
    - Partition count

------------------------------------------------------------
PHASE 1 OBJECTIVE
------------------------------------------------------------

Phase 1 provides a clean, modeling-ready feature store.

It intentionally excludes:

• Betting features
• PRA thresholds
• Hit/miss labels
• Modeling outputs
• Prediction logic

Those belong to Phase 2.

------------------------------------------------------------
NEXT STEP AFTER PHASE 1
------------------------------------------------------------

Migrate this notebook into a structured Python repository:

nba-feature-store/
    main.py
    config.py
    schema.py
    ingestion/
    utils/
    requirements.txt
    README.md

Colab serves as the prototype environment.
GitHub will host the production-ready version.

============================================================
PHASE 1 SCHEMA LOCKED
VERSION 1.0
============================================================
"""

'\nPHASE 1 ARCHITECTURE OVERVIEW\n============================================================\n\nThis notebook implements the Phase 1 version of the NBA\nFeature Store ingestion pipeline.\n\nThe schema and ingestion logic are now considered LOCKED\nfor Version 1.0. Any structural schema changes require\na version bump.\n\n------------------------------------------------------------\nTABLE STRUCTURE\n------------------------------------------------------------\n\nTable:\n    pr_see_daily_player_game_log\n\nPartitioning:\n    GAME_DATE (DATE)\n\nClustering:\n    PLAYER_ID\n    TEAM_ID\n\nPartition filtering is REQUIRED:\n    require_partition_filter = TRUE\n\nThis prevents accidental full-table scans and protects\nagainst runaway BigQuery costs.\n\n------------------------------------------------------------\nCOLUMN GROUPS\n------------------------------------------------------------\n\nIDENTITY\n    PLAYER_ID\n    PLAYER_NAME\n    TEAM_ID\n    TEAM_TRICODE\n    GAME_ID\n    ROW_KEY\n\n

In [7]:
# ============================================================
# SCHEMA DEFINITION — PHASE 1 LOCK (FULLY EXPLICIT)
# VERSION 1.0
# ============================================================

SCHEMA_DEFINITION = {

    # ---------------- IDENTITY ----------------
    "PLAYER_ID": "INT64",
    "PLAYER_NAME": "STRING",
    "TEAM_ID": "INT64",
    "TEAM_TRICODE": "STRING",
    "GAME_ID": "STRING",
    "ROW_KEY": "STRING",

    # ---------------- GAME CONTEXT ----------------
    "GAME_DATE": "DATE",
    "GAME_STATUS": "STRING",
    "GAME_TIME_UTC": "STRING",
    "ATTENDANCE": "INT64",
    "ARENA_NAME": "STRING",
    "ARENA_CITY": "STRING",
    "ARENA_STATE": "STRING",
    "ARENA_COUNTRY": "STRING",

    # ---------------- ROSTER ----------------
    "POSITION": "STRING",
    "HEIGHT": "STRING",
    "WEIGHT": "FLOAT64",
    "EXP": "STRING",

    # ---------------- MINUTES ----------------
    "minutes": "STRING",
    "minutes_SECONDS": "INT64",
    "DNP_FLAG": "BOOL",
    "GAMES_PLAYED_FLAG": "BOOL",

    # ---------------- TRADITIONAL ----------------
    "fieldGoalsMade": "INT64",
    "fieldGoalsAttempted": "INT64",
    "fieldGoalsPercentage": "FLOAT64",
    "threePointersMade": "INT64",
    "threePointersAttempted": "INT64",
    "threePointersPercentage": "FLOAT64",
    "freeThrowsMade": "INT64",
    "freeThrowsAttempted": "INT64",
    "freeThrowsPercentage": "FLOAT64",
    "reboundsOffensive": "INT64",
    "reboundsDefensive": "INT64",
    "reboundsTotal": "INT64",
    "assists": "INT64",
    "steals": "INT64",
    "blocks": "INT64",
    "turnovers": "INT64",
    "foulsPersonal": "INT64",
    "points": "INT64",
    "plusMinusPoints": "FLOAT64",

    # ---------------- ADVANCED ----------------
    "estimatedOffensiveRating_ADV": "FLOAT64",
    "offensiveRating_ADV": "FLOAT64",
    "estimatedDefensiveRating_ADV": "FLOAT64",
    "defensiveRating_ADV": "FLOAT64",
    "estimatedNetRating_ADV": "FLOAT64",
    "netRating_ADV": "FLOAT64",
    "assistPercentage_ADV": "FLOAT64",
    "assistToTurnover_ADV": "FLOAT64",
    "assistRatio_ADV": "FLOAT64",
    "offensiveReboundPercentage_ADV": "FLOAT64",
    "defensiveReboundPercentage_ADV": "FLOAT64",
    "reboundPercentage_ADV": "FLOAT64",
    "turnoverRatio_ADV": "FLOAT64",
    "effectiveFieldGoalPercentage_ADV": "FLOAT64",
    "trueShootingPercentage_ADV": "FLOAT64",
    "usagePercentage_ADV": "FLOAT64",
    "estimatedUsagePercentage_ADV": "FLOAT64",
    "estimatedPace_ADV": "FLOAT64",
    "pace_ADV": "FLOAT64",
    "pacePer40_ADV": "FLOAT64",
    "possessions_ADV": "FLOAT64",
    "PIE_ADV": "FLOAT64",

    # ---------------- USAGE ----------------
    "usagePercentage_USAGE": "FLOAT64",
    "percentageFieldGoalsMade_USAGE": "FLOAT64",
    "percentageFieldGoalsAttempted_USAGE": "FLOAT64",
    "percentageThreePointersMade_USAGE": "FLOAT64",
    "percentageThreePointersAttempted_USAGE": "FLOAT64",
    "percentageFreeThrowsMade_USAGE": "FLOAT64",
    "percentageFreeThrowsAttempted_USAGE": "FLOAT64",
    "percentageReboundsOffensive_USAGE": "FLOAT64",
    "percentageReboundsDefensive_USAGE": "FLOAT64",
    "percentageReboundsTotal_USAGE": "FLOAT64",
    "percentageAssists_USAGE": "FLOAT64",
    "percentageTurnovers_USAGE": "FLOAT64",
    "percentageSteals_USAGE": "FLOAT64",
    "percentageBlocks_USAGE": "FLOAT64",
    "percentageBlocksAllowed_USAGE": "FLOAT64",
    "percentagePersonalFouls_USAGE": "FLOAT64",
    "percentagePersonalFoulsDrawn_USAGE": "FLOAT64",
    "percentagePoints_USAGE": "FLOAT64",

    # ---------------- TRACKING ----------------
    "speed_TRACK": "FLOAT64",
    "distance_TRACK": "FLOAT64",
    "reboundChancesOffensive_TRACK": "INT64",
    "reboundChancesDefensive_TRACK": "INT64",
    "reboundChancesTotal_TRACK": "INT64",
    "touches_TRACK": "INT64",
    "secondaryAssists_TRACK": "INT64",
    "freeThrowAssists_TRACK": "INT64",
    "passes_TRACK": "INT64",
    "assists_TRACK": "INT64",
    "contestedFieldGoalsMade_TRACK": "INT64",
    "contestedFieldGoalsAttempted_TRACK": "INT64",
    "contestedFieldGoalPercentage_TRACK": "FLOAT64",
    "uncontestedFieldGoalsMade_TRACK": "INT64",
    "uncontestedFieldGoalsAttempted_TRACK": "INT64",
    "uncontestedFieldGoalsPercentage_TRACK": "FLOAT64",
    "fieldGoalPercentage_TRACK": "FLOAT64",
    "defendedAtRimFieldGoalsMade_TRACK": "INT64",
    "defendedAtRimFieldGoalsAttempted_TRACK": "INT64",
    "defendedAtRimFieldGoalPercentage_TRACK": "FLOAT64",

    # ---------------- TEAM CONTEXT ----------------
    "effectiveFieldGoalPercentage_TEAM": "FLOAT64",
    "freeThrowAttemptRate_TEAM": "FLOAT64",
    "teamTurnoverPercentage_TEAM": "FLOAT64",
    "offensiveReboundPercentage_TEAM": "FLOAT64",
    "oppEffectiveFieldGoalPercentage_TEAM": "FLOAT64",
    "oppFreeThrowAttemptRate_TEAM": "FLOAT64",
    "oppTeamTurnoverPercentage_TEAM": "FLOAT64",
    "oppOffensiveReboundPercentage_TEAM": "FLOAT64",

    # ---------------- SYSTEM ----------------
    "INGESTED_AT_UTC": "TIMESTAMP",
    "LOAD_BATCH_ID": "STRING",
}

In [8]:
# ============================================================
# SESSION SETUP
# (Persistent NBA API Session — Required for Stable Pulls)
# ============================================================

import requests
from nba_api.stats.library.http import NBAStatsHTTP

# Create persistent HTTP session
session = requests.Session()

session.headers.update({
    "Host": "stats.nba.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Connection": "keep-alive"
})

# Force nba_api to use this session globally
NBAStatsHTTP._session = session

print("[INFO] Persistent NBA session configured.")

[INFO] Persistent NBA session configured.


In [9]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

from datetime import datetime, timedelta
import time
import random
import requests


# ------------------------------------------------------------
# SIMPLE STRUCTURED LOGGER
# ------------------------------------------------------------

def log(level, message):
    """
    Basic structured logger for consistent console output.
    """
    print(f"[{level}] {message}")


# ------------------------------------------------------------
# DATE RANGE CONTROLLER
# ------------------------------------------------------------

def generate_date_list(start_date_str, end_date_str, exclude_dates=None):
    """
    Generates list of datetime.date objects between start and end (inclusive),
    removing any excluded dates.
    """

    if exclude_dates is None:
        exclude_dates = []

    # Validate date formats
    try:
        start_date = datetime.strptime(start_date_str, "%m/%d/%Y").date()
        end_date = datetime.strptime(end_date_str, "%m/%d/%Y").date()
    except ValueError:
        raise ValueError("START_DATE and END_DATE must be in MM/DD/YYYY format")

    if start_date > end_date:
        raise ValueError("START_DATE cannot be after END_DATE")

    # Build full date range
    all_dates = []
    current = start_date

    while current <= end_date:
        all_dates.append(current)
        current += timedelta(days=1)

    # Parse excluded dates
    exclude_dates_parsed = []
    for d in exclude_dates:
        try:
            exclude_dates_parsed.append(
                datetime.strptime(d, "%m/%d/%Y").date()
            )
        except ValueError:
            raise ValueError(f"Invalid date format in EXCLUDE_DATES: {d}")

    # Remove excluded dates
    final_dates = [d for d in all_dates if d not in exclude_dates_parsed]

    log("INFO", f"Generated {len(final_dates)} run dates.")
    return final_dates


# ------------------------------------------------------------
# MINUTES → SECONDS CONVERTER
# ------------------------------------------------------------

def minutes_to_seconds(val):
    """
    Converts 'MM:SS' string to total seconds.
    Returns 0 if value is null, empty, or malformed.
    """

    if isinstance(val, str) and ":" in val:
        try:
            minutes, seconds = val.split(":")
            return int(minutes) * 60 + int(seconds)
        except Exception:
            return 0

    return 0


# ------------------------------------------------------------
# RETRY WRAPPER — EXPONENTIAL BACKOFF + JITTER
# ------------------------------------------------------------

def call_with_retry(func, *args, max_retries=3, base_delay=1.0, **kwargs):
    """
    Executes a function with retry + exponential backoff + jitter.

    Parameters:
        func: callable
        max_retries: int
        base_delay: float (seconds)

    Behavior:
        - Retries up to max_retries
        - Backoff doubles each retry
        - Adds random jitter (0–0.5 sec)
        - Logs retry attempts
    """

    attempt = 0

    while attempt <= max_retries:
        try:
            return func(*args, **kwargs)

        except (requests.exceptions.RequestException, TimeoutError, ValueError) as e:

            if attempt == max_retries:
                log("ERROR", f"Max retries exceeded for {func.__name__}")
                raise

            sleep_time = (base_delay * (2 ** attempt)) + random.uniform(0, 0.5)

            log(
                "WARNING",
                f"{func.__name__} failed (attempt {attempt+1}/{max_retries}). "
                f"Retrying in {sleep_time:.2f}s..."
            )

            time.sleep(sleep_time)
            attempt += 1


# ------------------------------------------------------------
# VALIDATION LAYER
# ------------------------------------------------------------

def validate_daily_dataframe(df):

    log("INFO", "Running pre-ingestion validation checks...")

    # Required column checks
    required_columns = ["ROW_KEY", "PLAYER_ID", "minutes_SECONDS"]

    missing_required = [c for c in required_columns if c not in df.columns]
    if missing_required:
        raise ValueError(
            f"[VALIDATION ERROR] Missing required columns: {missing_required}"
        )

    # Duplicate ROW_KEY
    if df["ROW_KEY"].duplicated().any():
        dupes = df[df["ROW_KEY"].duplicated()]["ROW_KEY"].tolist()
        raise ValueError(
            f"[VALIDATION ERROR] Duplicate ROW_KEY detected. Examples: {dupes[:5]}"
        )

    # Duplicate columns
    if df.columns.duplicated().any():
        duplicate_cols = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(
            f"[VALIDATION ERROR] Duplicate column names detected: {duplicate_cols}"
        )

    # Null PLAYER_ID
    if df["PLAYER_ID"].isnull().any():
        null_count = df["PLAYER_ID"].isnull().sum()
        raise ValueError(
            f"[VALIDATION ERROR] {null_count} rows have NULL PLAYER_ID."
        )

    # Negative minutes
    if (df["minutes_SECONDS"] < 0).any():
        raise ValueError(
            "[VALIDATION ERROR] Negative minutes_SECONDS detected."
        )

    # Empty dataframe
    if len(df) == 0:
        raise ValueError(
            "[VALIDATION ERROR] Dataframe is empty before ingestion."
        )

    log("INFO", "All validation checks passed.")

In [10]:
# ============================================================
# RUN DATE GENERATION
# ============================================================

run_dates = generate_date_list(
    START_DATE,
    END_DATE,
    EXCLUDE_DATES
)

# ------------------------------------------------------------
# PRODUCTION SAFETY GUARDRAIL
# Prevent oversized historical backfills in a single run
# ------------------------------------------------------------

if len(run_dates) > MAX_DAYS_PER_RUN:
    raise ValueError(
        f"Requested {len(run_dates)} days. "
        f"Maximum allowed per run is {MAX_DAYS_PER_RUN}. "
        "Segment historical backfills into smaller windows."
    )

[INFO] Generated 1 run dates.


In [11]:
# ============================================================
# API PULL FUNCTIONS
# ============================================================

def pull_full_player_table(game_id):

    from nba_api.stats.endpoints import (
        boxscoretraditionalv3,
        boxscoreadvancedv3,
        boxscoreusagev3,
        boxscoreplayertrackv3
    )

    import pandas as pd
    import time

    # ============================================================
    # INTERNAL EXTRACTOR (STRICT + DETERMINISTIC)
    # ============================================================

    def extract_players(endpoint_obj):

        data = endpoint_obj.get_dict()
        main_key = [k for k in data.keys() if k.startswith("boxScore")][0]
        section = data[main_key]

        rows = []

        for side in ["homeTeam", "awayTeam"]:

            team = section.get(side, {})
            team_id = team.get("teamId")
            team_tricode = team.get("teamTricode")

            for player in team.get("players", []):

                first = player.get("firstName")
                last = player.get("familyName")
                display = player.get("displayName")

                if display:
                    player_name = display
                elif first and last:
                    player_name = f"{first} {last}"
                else:
                    player_name = None

                row = {
                    "PLAYER_ID": player.get("personId"),
                    "PLAYER_NAME": player_name,
                    "POSITION": player.get("positionFull") or player.get("position"),
                    "TEAM_ID": team_id,
                    "TEAM_TRICODE": team_tricode
                }

                stats = player.get("statistics", {})
                row.update(stats)

                rows.append(row)

        df = pd.DataFrame(rows)

        if "PLAYER_ID" in df.columns:
            df["PLAYER_ID"] = pd.to_numeric(df["PLAYER_ID"], errors="coerce")

        if "TEAM_ID" in df.columns:
            df["TEAM_ID"] = pd.to_numeric(df["TEAM_ID"], errors="coerce")

        return df


    # ============================================================
    # PULL ENDPOINTS (RETRY PROTECTED + PACED)
    # ============================================================

    traditional_raw = call_with_retry(
        boxscoretraditionalv3.BoxScoreTraditionalV3,
        game_id=game_id,
        timeout=45
    )
    traditional_df = extract_players(traditional_raw)

    time.sleep(0.6)

    advanced_raw = call_with_retry(
        boxscoreadvancedv3.BoxScoreAdvancedV3,
        game_id=game_id,
        timeout=45
    )
    advanced_df = extract_players(advanced_raw)

    time.sleep(0.6)

    usage_raw = call_with_retry(
        boxscoreusagev3.BoxScoreUsageV3,
        game_id=game_id,
        timeout=45
    )
    usage_df = extract_players(usage_raw)

    time.sleep(0.6)

    tracking_raw = call_with_retry(
        boxscoreplayertrackv3.BoxScorePlayerTrackV3,
        game_id=game_id,
        timeout=45
    )
    tracking_df = extract_players(tracking_raw)


    # ============================================================
    # LOCK IDENTITY — TRADITIONAL IS SOURCE OF TRUTH
    # ============================================================

    identity_cols = ["PLAYER_NAME", "POSITION", "TEAM_TRICODE"]

    advanced_df = advanced_df.drop(columns=identity_cols, errors="ignore")
    usage_df = usage_df.drop(columns=identity_cols, errors="ignore")
    tracking_df = tracking_df.drop(columns=identity_cols, errors="ignore")


    # ============================================================
    # RENAME SECONDARY STATS BEFORE MERGE
    # ============================================================

    def rename_stats(df, suffix):
        rename_map = {
            col: f"{col}{suffix}"
            for col in df.columns
            if col not in ["PLAYER_ID", "TEAM_ID"]
        }
        return df.rename(columns=rename_map)

    advanced_df = rename_stats(advanced_df, "_ADV")
    usage_df = rename_stats(usage_df, "_USAGE")
    tracking_df = rename_stats(tracking_df, "_TRACK")


    # ============================================================
    # STRICT MERGE (NO SUFFIX COLLISIONS POSSIBLE)
    # ============================================================

    df = (
        traditional_df
        .merge(
            advanced_df,
            on=["PLAYER_ID", "TEAM_ID"],
            how="left",
            validate="1:1"
        )
        .merge(
            usage_df,
            on=["PLAYER_ID", "TEAM_ID"],
            how="left",
            validate="1:1"
        )
        .merge(
            tracking_df,
            on=["PLAYER_ID", "TEAM_ID"],
            how="left",
            validate="1:1"
        )
    )

    # ============================================================
    # FINAL ENFORCEMENT
    # ============================================================

    df["GAME_ID"] = str(game_id)

    if df.columns.duplicated().any():
        raise ValueError("Duplicate columns detected inside pull_full_player_table.")

    return df

In [12]:
# ============================================================
# MULTI-DAY FEATURE STORE INGESTION LOOP
# (RETRY-HARDENED + PARTITION SAFE + IDEMPOTENT)
# + FULL ADAPTIVE GLOBAL RATE GOVERNOR
# + CHUNKED BACKFILL ENGINE
# + SESSION RESET PER BATCH (STABILITY FIX)
# + STRICT ATOMIC DAY ENFORCEMENT
# ============================================================

from nba_api.stats.endpoints import (
    scoreboardv3,
    commonteamroster,
    boxscorefourfactorsv3,
    boxscoresummaryv3
)
from nba_api.stats.library.http import NBAStatsHTTP
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from collections import deque
import requests
import time
import pandas as pd


# ============================================================
# SESSION RESET (PER BATCH STABILITY FIX)
# ============================================================

def reset_nba_session():

    session = requests.Session()

    session.headers.update({
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nba.com/",
        "Origin": "https://www.nba.com",
        "Connection": "keep-alive"
    })

    NBAStatsHTTP._session = session
    log("INFO", "NBA session reset for new batch.")


# ============================================================
# FULL RATE GOVERNOR (UNCHANGED)
# ============================================================

class RateGovernor:

    def __init__(self):

        self.window_seconds = 300
        self.request_timestamps = deque()

        self.throttle_level = 0
        self.max_throttle_level = 4

        self.multipliers = [1.0, 1.5, 2.0, 3.0, 4.0]

        self.base_endpoint_delay = 0.6
        self.base_game_delay = 0.9
        self.base_day_delay = 3.0

        self.failure_score = 0
        self.consecutive_failures = 0

        self.cooldown_tiers = [60, 120, 240, 300]
        self.cooldown_index = 0

        self.batch_cooldown = 120
        self.max_batch_cooldown = 300

        self.max_requests_per_minute = 75

    def register_request(self):

        now = time.time()
        self.request_timestamps.append(now)

        while self.request_timestamps and \
              now - self.request_timestamps[0] > self.window_seconds:
            self.request_timestamps.popleft()

        self._evaluate_pressure()

    def _evaluate_pressure(self):

        rpm = len(self.request_timestamps) / (self.window_seconds / 60)

        if rpm > self.max_requests_per_minute:
            self._increase_throttle()

    def _increase_throttle(self):
        if self.throttle_level < self.max_throttle_level:
            self.throttle_level += 1
            log("WARNING", f"Throttle level increased to {self.throttle_level}")

    def _decrease_throttle(self):
        if self.throttle_level > 0:
            self.throttle_level -= 1
            log("INFO", f"Throttle level decreased to {self.throttle_level}")

    def multiplier(self):
        return self.multipliers[self.throttle_level]

    def sleep_endpoint(self):
        time.sleep(self.base_endpoint_delay * self.multiplier())

    def sleep_game(self):
        time.sleep(self.base_game_delay * self.multiplier())

    def sleep_day(self):
        time.sleep(self.base_day_delay * self.multiplier())

    def record_success(self):

        self.consecutive_failures = 0

        if self.failure_score > 0:
            self.failure_score -= 1

        if self.failure_score == 0:
            self._decrease_throttle()

    def record_failure(self):

        self.failure_score += 1
        self.consecutive_failures += 1
        self._increase_throttle()

    def apply_cooldown_if_needed(self):

        if self.consecutive_failures >= 2:

            cooldown = self.cooldown_tiers[self.cooldown_index]

            log("WARNING", f"Applying cooldown: {cooldown}s")
            time.sleep(cooldown)

            if self.cooldown_index < len(self.cooldown_tiers) - 1:
                self.cooldown_index += 1

            self.consecutive_failures = 0

    def sleep_batch(self):

        log("INFO", f"Batch cooldown: {self.batch_cooldown}s")
        time.sleep(self.batch_cooldown)

        self.batch_cooldown = min(
            int(self.batch_cooldown * 1.5),
            self.max_batch_cooldown
        )


# ============================================================
# INITIALIZATION
# ============================================================

governor = RateGovernor()

CHUNK_SIZE_DAYS = 7

log("INFO", f"Total run dates: {len(run_dates)}")
log("INFO", f"Batch size: {CHUNK_SIZE_DAYS} days")

# ============================================================
# RUN TRACKING (SUCCESS / FAILURE VISIBILITY)
# ============================================================

failed_days = []
successful_days = []


# ============================================================
# CHUNKED BATCH LOOP
# ============================================================

for batch_start in range(0, len(run_dates), CHUNK_SIZE_DAYS):

    reset_nba_session()

    batch = run_dates[batch_start: batch_start + CHUNK_SIZE_DAYS]
    batch_number = (batch_start // CHUNK_SIZE_DAYS) + 1

    log("INFO", f"========== STARTING BATCH {batch_number} ==========")

    for run_date in batch:

        run_date_str = run_date.strftime("%m/%d/%Y")
        log("INFO", f"Starting processing for {run_date_str}")

        try:

            governor.register_request()
            scoreboard = call_with_retry(
                scoreboardv3.ScoreboardV3,
                game_date=run_date_str,
                timeout=30
            )

            games = scoreboard.get_dict()["scoreboard"]["games"]
            games_df = pd.DataFrame(games)

            if len(games_df) == 0:
                log("WARNING", f"No games found for {run_date_str}. Skipping.")
                governor.record_success()
                governor.sleep_day()
                continue

            games_df["GAME_ID"] = games_df["gameId"]
            games_df["HOME_TEAM_ID"] = games_df["homeTeam"].apply(lambda x: x["teamId"])
            games_df["VISITOR_TEAM_ID"] = games_df["awayTeam"].apply(lambda x: x["teamId"])

            games_df = games_df[["GAME_ID","HOME_TEAM_ID","VISITOR_TEAM_ID"]]

            all_game_logs = []
            game_ids = games_df["GAME_ID"].astype(str).tolist()

            log("INFO", f"Processing {len(game_ids)} games for {run_date_str}")

            columns_to_drop = [
                "speed","distance",
                "reboundChancesOffensive","reboundChancesDefensive","reboundChancesTotal",
                "touches","secondaryAssists","freeThrowAssists","passes",
                "contestedFieldGoalsMade","contestedFieldGoalsAttempted",
                "contestedFieldGoalPercentage","uncontestedFieldGoalsMade",
                "uncontestedFieldGoalsAttempted","uncontestedFieldGoalsPercentage",
                "defendedAtRimFieldGoalsMade","defendedAtRimFieldGoalsAttempted",
                "defendedAtRimFieldGoalPercentage"
            ]

            day_failed = False

            for game_id in game_ids:

                log("INFO", f"Processing game {game_id}")

                try:

                    governor.register_request()

                    player_game_log = call_with_retry(
                        pull_full_player_table,
                        game_id
                    )

                    player_game_log["GAME_ID"] = str(game_id)

                    player_game_log["PLAYER_ID"] = pd.to_numeric(
                        player_game_log["PLAYER_ID"], errors="coerce"
                    ).astype("Int64")

                    player_game_log["TEAM_ID"] = pd.to_numeric(
                        player_game_log["TEAM_ID"], errors="coerce"
                    ).astype("Int64")

                    player_game_log = player_game_log.drop(
                        columns=["POSITION"], errors="ignore"
                    )

                    if player_game_log.columns.duplicated().any():
                        raise ValueError(f"Duplicate columns in base table for {game_id}")

                    game_row = games_df.loc[
                        games_df["GAME_ID"].astype(str) == game_id
                    ].iloc[0]

                    team_ids = [
                        int(game_row["HOME_TEAM_ID"]),
                        int(game_row["VISITOR_TEAM_ID"])
                    ]

                    roster_frames = []

                    for team_id in team_ids:

                        governor.register_request()
                        roster_endpoint = call_with_retry(
                            commonteamroster.CommonTeamRoster,
                            team_id=team_id,
                            timeout=45
                        )

                        roster_df = roster_endpoint.get_data_frames()[0]

                        roster_df["PLAYER_ID"] = pd.to_numeric(
                            roster_df["PLAYER_ID"], errors="coerce"
                        ).astype("Int64")

                        roster_frames.append(
                            roster_df[["PLAYER_ID","POSITION","HEIGHT","WEIGHT","EXP"]]
                        )

                        governor.sleep_endpoint()

                    roster_clean = (
                        pd.concat(roster_frames, ignore_index=True)
                        .drop_duplicates("PLAYER_ID")
                    )

                    roster_clean["WEIGHT"] = pd.to_numeric(
                        roster_clean["WEIGHT"], errors="coerce"
                    )
                    roster_clean["HEIGHT"] = roster_clean["HEIGHT"].astype(str)
                    roster_clean["EXP"] = roster_clean["EXP"].astype(str)

                    player_game_log = player_game_log.merge(
                        roster_clean,
                        on="PLAYER_ID",
                        how="left",
                        validate="m:1"
                    )

                    governor.register_request()
                    four = call_with_retry(
                        boxscorefourfactorsv3.BoxScoreFourFactorsV3,
                        game_id=game_id,
                        timeout=45
                    )

                    four_data = four.get_dict()["boxScoreFourFactors"]

                    rows = []
                    for side in ["homeTeam","awayTeam"]:
                        team = four_data[side]
                        row = {"TEAM_ID": team.get("teamId")}
                        row.update(team.get("statistics", {}))
                        rows.append(row)

                    team_four_df = pd.DataFrame(rows)
                    team_four_df["TEAM_ID"] = pd.to_numeric(
                        team_four_df["TEAM_ID"], errors="coerce"
                    ).astype("Int64")
                    team_four_df["GAME_ID"] = str(game_id)
                    team_four_df = team_four_df.drop(columns=["minutes"], errors="ignore")

                    player_game_log = player_game_log.merge(
                        team_four_df,
                        on=["TEAM_ID","GAME_ID"],
                        how="left",
                        validate="m:1"
                    )

                    team_rename_map = {
                        "effectiveFieldGoalPercentage": "effectiveFieldGoalPercentage_TEAM",
                        "freeThrowAttemptRate": "freeThrowAttemptRate_TEAM",
                        "teamTurnoverPercentage": "teamTurnoverPercentage_TEAM",
                        "offensiveReboundPercentage": "offensiveReboundPercentage_TEAM",
                        "oppEffectiveFieldGoalPercentage": "oppEffectiveFieldGoalPercentage_TEAM",
                        "oppFreeThrowAttemptRate": "oppFreeThrowAttemptRate_TEAM",
                        "oppTeamTurnoverPercentage": "oppTeamTurnoverPercentage_TEAM",
                        "oppOffensiveReboundPercentage": "oppOffensiveReboundPercentage_TEAM"
                    }

                    player_game_log = player_game_log.rename(
                        columns={k: v for k, v in team_rename_map.items()
                                 if k in player_game_log.columns}
                    )

                    governor.register_request()
                    summary = call_with_retry(
                        boxscoresummaryv3.BoxScoreSummaryV3,
                        game_id=game_id,
                        timeout=45
                    )

                    summary_data = summary.get_dict()["boxScoreSummary"]

                    meta_df = pd.DataFrame([{
                        "GAME_ID": str(summary_data["gameId"]),
                        "GAME_STATUS": summary_data["gameStatusText"],
                        "GAME_TIME_UTC": summary_data["gameTimeUTC"],
                        "ATTENDANCE": summary_data.get("attendance"),
                        "ARENA_NAME": summary_data["arena"]["arenaName"],
                        "ARENA_CITY": summary_data["arena"]["arenaCity"],
                        "ARENA_STATE": summary_data["arena"]["arenaState"],
                        "ARENA_COUNTRY": summary_data["arena"]["arenaCountry"]
                    }])

                    player_game_log = player_game_log.merge(
                        meta_df,
                        on="GAME_ID",
                        how="left",
                        validate="m:1"
                    )

                    player_game_log = player_game_log.drop(
                        columns=[c for c in columns_to_drop if c in player_game_log.columns],
                        errors="ignore"
                    )

                    all_game_logs.append(player_game_log)

                    governor.sleep_game()

                except Exception as game_error:

                    log(
                        "ERROR",
                        f"Game {game_id} failed after retries. "
                        f"Aborting entire day {run_date_str}."
                    )

                    day_failed = True
                    break

            if day_failed or len(all_game_logs) != len(game_ids):

                raise ValueError(
                    f"Day {run_date_str} incomplete. "
                    f"Expected {len(game_ids)} games, "
                    f"processed {len(all_game_logs)}."
                )

            daily_player_game_log = pd.concat(all_game_logs, ignore_index=True)

            daily_player_game_log["minutes"] = (
                daily_player_game_log.get("minutes", "0:00")
                .fillna("0:00")
                .replace("", "0:00")
            )

            daily_player_game_log["minutes_SECONDS"] = (
                daily_player_game_log["minutes"].apply(minutes_to_seconds)
            )

            daily_player_game_log["DNP_FLAG"] = (
                daily_player_game_log["minutes_SECONDS"] == 0
            )

            daily_player_game_log["GAMES_PLAYED_FLAG"] = (
                daily_player_game_log["minutes_SECONDS"] > 0
            )

            daily_player_game_log["ROW_KEY"] = (
                daily_player_game_log["GAME_ID"].astype(str)
                + "_"
                + daily_player_game_log["PLAYER_ID"].astype(str)
            )

            ingest_time = pd.Timestamp.utcnow()
            load_batch_id = ingest_time.strftime("%Y%m%d%H%M%S")

            daily_player_game_log["GAME_DATE"] = run_date
            daily_player_game_log["INGESTED_AT_UTC"] = ingest_time
            daily_player_game_log["LOAD_BATCH_ID"] = load_batch_id

            validate_daily_dataframe(daily_player_game_log)

            expected_columns = list(SCHEMA_DEFINITION.keys())
            daily_player_game_log = daily_player_game_log[expected_columns]

            try:
                table = client.get_table(TABLE_ID)
            except NotFound:

                schema = [
                    bigquery.SchemaField(col, dtype)
                    for col, dtype in SCHEMA_DEFINITION.items()
                ]

                table = bigquery.Table(TABLE_ID, schema=schema)
                table.time_partitioning = bigquery.TimePartitioning(field="GAME_DATE")
                table.clustering_fields = ["PLAYER_ID", "TEAM_ID"]

                client.create_table(table)

                client.query(f"""
                    ALTER TABLE `{TABLE_ID}`
                    SET OPTIONS (require_partition_filter = TRUE)
                """).result()

            client.query(f"""
                DELETE FROM `{TABLE_ID}`
                WHERE GAME_DATE = DATE('{run_date}')
            """).result()

            job = client.load_table_from_dataframe(
                daily_player_game_log,
                TABLE_ID,
                job_config=bigquery.LoadJobConfig(
                    write_disposition="WRITE_APPEND"
                )
            )

            job.result()

            result_df = client.query(f"""
                SELECT COUNT(*) AS row_count
                FROM `{TABLE_ID}`
                WHERE GAME_DATE = DATE('{run_date}')
            """).to_dataframe()

            inserted_rows = result_df["row_count"].iloc[0]

            if inserted_rows != len(daily_player_game_log):
                raise ValueError(
                    f"Row count mismatch for {run_date}: "
                    f"expected {len(daily_player_game_log)}, "
                    f"found {inserted_rows}"
                )

            log("INFO", f"Completed {run_date_str} — {inserted_rows} rows inserted")

            successful_days.append(run_date_str)

            governor.record_success()
            governor.sleep_day()

        except Exception as e:

            log("ERROR", f"Failure on {run_date_str}: {e}")

            failed_days.append(run_date_str)

            governor.record_failure()
            governor.apply_cooldown_if_needed()
            continue

    log("INFO", f"Batch {batch_number} complete.")
    governor.sleep_batch()

log("INFO", "Multi-day ingestion complete.")

# ============================================================
# RUN SUMMARY
# ============================================================

print("\n================ RUN SUMMARY ================\n")

print(f"Successful days ({len(successful_days)}):")
for d in successful_days:
    print(f"  ✓ {d}")

print(f"\nFailed days ({len(failed_days)}):")
for d in failed_days:
    print(f"  ✗ {d}")

print("\n=============================================\n")

[INFO] Total run dates: 1
[INFO] Batch size: 7 days
[INFO] NBA session reset for new batch.
[INFO] ========== STARTING BATCH 1 ==========
[INFO] Starting processing for 03/03/2026
[INFO] Processing 10 games for 03/03/2026
[INFO] Processing game 0022500883
[INFO] Processing game 0022500884
[INFO] Processing game 0022500885
[INFO] Processing game 0022500886
[INFO] Processing game 0022500887
[INFO] Processing game 0022500888
[INFO] Processing game 0022500889
[INFO] Processing game 0022500890
[INFO] Processing game 0022500891
[INFO] Processing game 0022500892
[INFO] Running pre-ingestion validation checks...
[INFO] All validation checks passed.
[INFO] Completed 03/03/2026 — 267 rows inserted
[INFO] Batch 1 complete.
[INFO] Batch cooldown: 120s
[INFO] Multi-day ingestion complete.

================ RUN SUMMARY ================

Successful days (1):
  ✓ 03/03/2026

Failed days (0):




In [20]:
# ============================================================
# DATA HEALTH AUDIT — REGULAR SEASON COMPLETENESS CHECK
# ============================================================
# PURPOSE
# ------------------------------------------------------------
# This cell audits the feature store for:
#
# • Missing regular season game dates
# • Duplicate ROW_KEY values
# • Daily row counts
#
# It does NOT modify data.
# It only reads from BigQuery.
#
# Designed for quick visual validation that the ingestion
# pipeline is functioning correctly.
#
# IMPORTANT TIME LOGIC
# ------------------------------------------------------------
# The ingestion pipeline runs for YESTERDAY's NBA games.
#
# Therefore this audit evaluates:
#
#     SEASON_START → Yesterday (US/Eastern)
#
# Using Eastern time prevents today's games from appearing
# as missing when UTC rolls into the next day while it is
# still the prior calendar day locally.
#
# Known NBA "no game days" are excluded to prevent false flags.
# ============================================================

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd


# ============================================================
# SEASON CONFIGURATION
# ============================================================

SEASON_START = "2025-10-21"
SEASON_END   = "2026-04-12"

ALL_STAR_BREAK = [
    "2026-02-13",
    "2026-02-14",
    "2026-02-15",
    "2026-02-16",
    "2026-02-17",
    "2026-02-18",
]

# Known NBA off days (no scheduled games)
KNOWN_NO_GAME_DAYS = [
    "2025-11-27",  # Thanksgiving
    "2025-12-24",  # Christmas Eve
]


# ============================================================
# DETERMINE AUDIT WINDOW (EASTERN TIME)
# ============================================================

season_start_dt = datetime.strptime(SEASON_START, "%Y-%m-%d").date()
season_end_dt   = datetime.strptime(SEASON_END, "%Y-%m-%d").date()

eastern_now = datetime.now(ZoneInfo("America/New_York")).date()

yesterday_eastern = eastern_now - timedelta(days=1)

audit_end_dt = min(yesterday_eastern, season_end_dt)

audit_end_str = audit_end_dt.strftime("%Y-%m-%d")


# ============================================================
# GENERATE EXPECTED REGULAR SEASON CALENDAR
# ============================================================

expected_dates = []

current = season_start_dt

while current <= audit_end_dt:

    date_str = current.strftime("%Y-%m-%d")

    if (
        date_str not in ALL_STAR_BREAK
        and date_str not in KNOWN_NO_GAME_DAYS
    ):
        expected_dates.append(date_str)

    current += timedelta(days=1)

expected_df = pd.DataFrame({"GAME_DATE": expected_dates})


# ============================================================
# QUERY BIGQUERY DAILY HEALTH
# ============================================================

audit_query = f"""
SELECT
    GAME_DATE,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ROW_KEY) AS distinct_row_keys,
    COUNT(*) - COUNT(DISTINCT ROW_KEY) AS duplicate_row_keys
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{SEASON_START}') AND DATE('{audit_end_str}')
GROUP BY GAME_DATE
ORDER BY GAME_DATE
"""

audit_df = client.query(audit_query).to_dataframe()

audit_df["GAME_DATE"] = audit_df["GAME_DATE"].astype(str)


# ============================================================
# FIND MISSING DAYS
# ============================================================

merged = expected_df.merge(
    audit_df,
    on="GAME_DATE",
    how="left"
)

missing_days = merged[merged["total_rows"].isna()]


# ============================================================
# FIND DUPLICATE ROW KEYS
# ============================================================

duplicate_days = audit_df[audit_df["duplicate_row_keys"] > 0]


# ============================================================
# OUTPUT RESULTS
# ============================================================

print("\n================ DATA HEALTH AUDIT ================\n")

print(f"Current Eastern Date : {eastern_now}")
print("AUDIT WINDOW:")
print(f"Season Start : {SEASON_START}")
print(f"Audit End    : {audit_end_str}")

print("\n--------------------------------------------------")
print("Daily Table Status")
print("--------------------------------------------------")

display(audit_df)

print("\n--------------------------------------------------")
print("Missing Regular Season Dates")
print("--------------------------------------------------")

if len(missing_days) == 0:
    print("✓ No missing dates detected.")
else:
    for d in missing_days["GAME_DATE"]:
        print(f"✗ {d}")

print("\n--------------------------------------------------")
print("Duplicate ROW_KEY Issues")
print("--------------------------------------------------")

if len(duplicate_days) == 0:
    print("✓ No duplicate ROW_KEY values detected.")
else:
    display(duplicate_days)

print("\n==================================================\n")


================ DATA HEALTH AUDIT ================

Current Eastern Date : 2026-03-04
AUDIT WINDOW:
Season Start : 2025-10-21
Audit End    : 2026-03-03

--------------------------------------------------
Daily Table Status
--------------------------------------------------


,GAME_DATE,total_rows,distinct_row_keys,duplicate_row_keys
0,2025-10-21,52,52,0
1,2025-10-22,334,334,0
2,2025-10-23,53,53,0
3,2025-10-24,329,329,0
4,2025-10-25,129,129,0
...,...,...,...,...
121,2026-02-27,133,133,0
122,2026-02-28,129,129,0
123,2026-03-01,287,287,0
124,2026-03-02,104,104,0



--------------------------------------------------
Missing Regular Season Dates
--------------------------------------------------
✓ No missing dates detected.

--------------------------------------------------
Duplicate ROW_KEY Issues
--------------------------------------------------
✓ No duplicate ROW_KEY values detected.




In [21]:
# ============================================================
# GAME INTEGRITY AUDIT — FEATURE STORE VALIDATION
# ============================================================
# PURPOSE
# ------------------------------------------------------------
# Validates that every NBA game ingested into the feature
# store appears structurally complete.
#
# This audit checks for:
#
# • Games missing one of the two teams
# • Teams with abnormally low player counts
# • Games with suspiciously low total player counts
# • Corrupted merges (NULL player/team IDs)
#
# IMPORTANT TIME LOGIC
# ------------------------------------------------------------
# The ingestion pipeline runs for YESTERDAY's NBA games.
#
# Therefore this audit evaluates:
#
#     SEASON_START → Yesterday (US/Eastern)
#
# Using Eastern time prevents false flags when UTC rolls
# into the next day while it is still the prior calendar
# day locally.
#
# BigQuery partition filters are applied to respect
# require_partition_filter = TRUE.
# ============================================================

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd


# ============================================================
# DETERMINE AUDIT WINDOW (EASTERN DATE ONLY)
# ============================================================

season_start = "2025-10-21"

eastern_today = datetime.now(ZoneInfo("America/New_York")).date()

yesterday_eastern = eastern_today - timedelta(days=1)

audit_end = yesterday_eastern.strftime("%Y-%m-%d")


# ============================================================
# GAME STRUCTURE CHECK
# ============================================================

game_structure_query = f"""
SELECT
    GAME_ID,
    COUNT(DISTINCT TEAM_ID) AS teams_in_game,
    COUNT(*) AS rows_in_game
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{season_start}') AND DATE('{audit_end}')
GROUP BY GAME_ID
ORDER BY GAME_ID
"""

game_structure = client.query(game_structure_query).to_dataframe()


# ============================================================
# TEAM PLAYER COUNT CHECK
# ============================================================

team_player_query = f"""
SELECT
    GAME_ID,
    TEAM_ID,
    COUNT(*) AS players_logged
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{season_start}') AND DATE('{audit_end}')
GROUP BY GAME_ID, TEAM_ID
ORDER BY GAME_ID
"""

team_players = client.query(team_player_query).to_dataframe()


# ============================================================
# GAME PLAYER COUNT CHECK (NEW)
# ============================================================

game_player_query = f"""
SELECT
    GAME_ID,
    COUNT(*) AS players_in_game
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{season_start}') AND DATE('{audit_end}')
GROUP BY GAME_ID
ORDER BY GAME_ID
"""

game_players = client.query(game_player_query).to_dataframe()


# ============================================================
# CORRUPTED ROW CHECK
# ============================================================

null_check_query = f"""
SELECT
    COUNT(*) AS corrupted_rows
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{season_start}') AND DATE('{audit_end}')
AND (
    PLAYER_ID IS NULL
    OR TEAM_ID IS NULL
    OR GAME_ID IS NULL
)
"""

null_check = client.query(null_check_query).to_dataframe()


# ============================================================
# FIND PROBLEM GAMES
# ============================================================

bad_team_games = game_structure[game_structure["teams_in_game"] != 2]

low_player_teams = team_players[team_players["players_logged"] < 5]

# New protection: games with suspiciously low player totals
suspicious_games = game_players[game_players["players_in_game"] < 16]

corrupted_rows = null_check["corrupted_rows"].iloc[0]


# ============================================================
# OUTPUT RESULTS
# ============================================================

print("\n================ GAME INTEGRITY AUDIT ================\n")

print(f"Current Eastern Date : {eastern_today}")
print(f"Audit Window End     : {audit_end}")

print("\n--------------------------------------------------")
print("Games Missing Teams")
print("--------------------------------------------------")

if len(bad_team_games) == 0:
    print("✓ All games contain exactly two teams.")
else:
    display(bad_team_games)


print("\n--------------------------------------------------")
print("Teams With Suspiciously Low Player Counts")
print("--------------------------------------------------")

if len(low_player_teams) == 0:
    print("✓ All teams have reasonable player counts.")
else:
    display(low_player_teams)


print("\n--------------------------------------------------")
print("Games With Suspiciously Low Total Player Counts")
print("--------------------------------------------------")

if len(suspicious_games) == 0:
    print("✓ All games contain reasonable total player counts.")
else:
    display(suspicious_games)


print("\n--------------------------------------------------")
print("Corrupted Row Check")
print("--------------------------------------------------")

if corrupted_rows == 0:
    print("✓ No corrupted rows detected.")
else:
    print(f"⚠ {corrupted_rows} corrupted rows detected.")


print("\n====================================================\n")


================ GAME INTEGRITY AUDIT ================

Current Eastern Date : 2026-03-04
Audit Window End     : 2026-03-03

--------------------------------------------------
Games Missing Teams
--------------------------------------------------
✓ All games contain exactly two teams.

--------------------------------------------------
Teams With Suspiciously Low Player Counts
--------------------------------------------------
✓ All teams have reasonable player counts.

--------------------------------------------------
Games With Suspiciously Low Total Player Counts
--------------------------------------------------
✓ All games contain reasonable total player counts.

--------------------------------------------------
Corrupted Row Check
--------------------------------------------------
✓ No corrupted rows detected.




In [22]:
# ============================================================
# FEATURE STORE COMMAND CENTER
# PIPELINE HEALTH DASHBOARD
# ============================================================
# PURPOSE
# ------------------------------------------------------------
# Provides a one-screen overview of the health of the
# NBA Feature Store.
#
# This combines several operational checks:
#
# • Freshness of ingestion
# • Total rows in the feature store
# • Total games ingested
# • Unique players observed
# • Number of partitions (game dates)
#
# IMPORTANT TIME LOGIC
# ------------------------------------------------------------
# The ingestion pipeline runs for YESTERDAY's NBA games.
#
# Expected latest partition = Yesterday (US/Eastern)
#
# Using Eastern date prevents false "days behind" flags
# when UTC rolls into the next day while it is still the
# prior calendar day locally.
#
# This cell is READ ONLY and safe to run anytime.
#
# BigQuery partition filters are respected to ensure
# low query cost.
# ============================================================

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd


# ============================================================
# SEASON CONFIGURATION
# ============================================================

SEASON_START = "2025-10-21"


# ============================================================
# DETERMINE HEALTH WINDOW (EASTERN DATE)
# ============================================================

eastern_today = datetime.now(ZoneInfo("America/New_York")).date()

yesterday_eastern = eastern_today - timedelta(days=1)

audit_end = yesterday_eastern.strftime("%Y-%m-%d")


# ============================================================
# LAST INGESTED GAME DATE
# ============================================================

last_ingest_query = f"""
SELECT
    MAX(GAME_DATE) AS last_game_date
FROM `{TABLE_ID}`
WHERE GAME_DATE >= DATE('{SEASON_START}')
"""

last_ingest_df = client.query(last_ingest_query).to_dataframe()

last_ingest_date = str(last_ingest_df["last_game_date"].iloc[0])


# ============================================================
# TOTAL PLAYER ROWS
# ============================================================

row_count_query = f"""
SELECT
    COUNT(*) AS total_rows
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{SEASON_START}') AND DATE('{audit_end}')
"""

rows_df = client.query(row_count_query).to_dataframe()

total_rows = rows_df["total_rows"].iloc[0]


# ============================================================
# TOTAL GAMES INGESTED
# ============================================================

games_query = f"""
SELECT
    COUNT(DISTINCT GAME_ID) AS total_games
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{SEASON_START}') AND DATE('{audit_end}')
"""

games_df = client.query(games_query).to_dataframe()

total_games = games_df["total_games"].iloc[0]


# ============================================================
# UNIQUE PLAYERS
# ============================================================

players_query = f"""
SELECT
    COUNT(DISTINCT PLAYER_ID) AS unique_players
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{SEASON_START}') AND DATE('{audit_end}')
"""

players_df = client.query(players_query).to_dataframe()

unique_players = players_df["unique_players"].iloc[0]


# ============================================================
# PARTITION COUNT
# ============================================================

partition_query = f"""
SELECT
    COUNT(DISTINCT GAME_DATE) AS partition_count
FROM `{TABLE_ID}`
WHERE GAME_DATE BETWEEN DATE('{SEASON_START}') AND DATE('{audit_end}')
"""

partition_df = client.query(partition_query).to_dataframe()

partition_count = partition_df["partition_count"].iloc[0]


# ============================================================
# PIPELINE FRESHNESS
# ============================================================

expected_latest = audit_end

days_behind = (
    pd.to_datetime(expected_latest) - pd.to_datetime(last_ingest_date)
).days


# ============================================================
# OUTPUT DASHBOARD
# ============================================================

print("\n================ FEATURE STORE HEALTH ================\n")

print(f"Current Eastern Date : {eastern_today}")
print(f"Last Game Ingested   : {last_ingest_date}")
print(f"Expected Latest      : {expected_latest}")
print(f"Days Behind          : {days_behind}")

print("\n--------------------------------------------------")

print(f"Total Player Rows    : {total_rows:,}")
print(f"Total Games          : {total_games:,}")
print(f"Unique Players       : {unique_players:,}")
print(f"Table Partitions     : {partition_count:,}")

print("\n=====================================================\n")


================ FEATURE STORE HEALTH ================

Current Eastern Date : 2026-03-04
Last Game Ingested   : 2026-03-03
Expected Latest      : 2026-03-03
Days Behind          : 0

--------------------------------------------------
Total Player Rows    : 24,007
Total Games          : 921
Unique Players       : 559
Table Partitions     : 126




#End of Phase 1 - Loading Game/ Player Data + Logging in BigQuery
    Currently runs and logs 10/21/2025 to 03/03/2026
    to daily_player_game_log